In [6]:


# ==========================================
# 1. 5-LINE PACKAGE INSTALLATION BLOCK
# ==========================================
!pip install -q torch torchvision numpy pandas pillow scikit-learn
!pip install -q matplotlib seaborn
# ==========================================

# ==========================================
# 2. LENET CNN DEEP LEARNING AUTOML CODE
# ==========================================
import os
import time
import zipfile
import json
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

# Colab Uploads / Downloads
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- CONFIGURABLE HYPERPARAMETERS ---
ZIP_LIMIT_MB = 500
DATA_DIR = Path("./bengali_character_data") if not IN_COLAB else Path("/content/bengali_character_data")
IMAGE_EXTENSIONS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}

# CNN Hyperparameters
IMAGE_SIZE = 32  # LeNet input size
BATCH_SIZE = 64
EPOCHS = 25
LEARNING_RATE = 0.001
RANDOM_STATE = 42

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using Device: {device} ({'GPU Acceleration Active!' if device.type == 'cuda' else 'CPU Mode - GPU recommended!'})")


def upload_and_extract_zip() -> Path:
    """Handles the uploading and extraction of the zip file in Colab or locally."""
    if IN_COLAB:
        print("💡 [STEP 1] Upload your dataset zip file...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("❌ No file uploaded.")
        zip_name = next(iter(uploaded.keys()))
        zip_path = Path("/content") / zip_name
    else:
        print("💡 [STEP 1] Running locally. Looking for dataset.zip in current directory...")
        zip_path = Path("dataset.zip")
        if not zip_path.exists():
            zips = list(Path(".").glob("*.zip"))
            if zips:
                zip_path = zips[0]
                print(f"Found zip: {zip_path}")
            else:
                raise FileNotFoundError("❌ Please place a 'dataset.zip' file in the current directory.")

    size_mb = zip_path.stat().st_size / (1024 * 1024)
    if size_mb > ZIP_LIMIT_MB:
        raise ValueError(f"❌ Zip file size ({size_mb:.1f} MB) exceeds the limit of {ZIP_LIMIT_MB} MB.")

    print(f"🔄 Extracting dataset to {DATA_DIR.resolve()}...")
    if DATA_DIR.exists():
        import shutil
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (DATA_DIR / member.filename).resolve()
            if not str(target).startswith(str(DATA_DIR.resolve())):
                raise ValueError(f"❌ Unsafe zip path found: {member.filename}")
        archive.extractall(DATA_DIR)
    print("✅ Extraction complete.")
    return DATA_DIR


class BengaliDataset(Dataset):
    """Custom PyTorch dataset to load and preprocess handwritten character images."""
    def __init__(self, folder_path: Path, class_names: List[str], class_to_idx: Dict[str, int]):
        self.samples = []
        self.class_to_idx = class_to_idx

        for class_name in class_names:
            class_folder = folder_path / class_name
            if not class_folder.exists():
                continue
            idx = class_to_idx[class_name]
            for file_path in class_folder.iterdir():
                if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                    self.samples.append((file_path, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, item):
        file_path, label = self.samples[item]
        with Image.open(file_path) as img:
            img = img.convert("L")
            img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
            arr = np.array(img, dtype=np.float32) / 255.0
            tensor = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
            tensor = (tensor - 0.5) / 0.5
            return tensor, label


class SuperChargedLeNet(nn.Module):
    """Modernized LeNet model designed for exceptionally high classification accuracies."""
    def __init__(self, num_classes: int):
        super(SuperChargedLeNet, self).__init__()

        # Layer 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout2d(0.15)

        # Layer 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=0)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout2d(0.2)

        # Layer 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.drop3 = nn.Dropout2d(0.25)

        # Linear Layers
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.bn_fc1 = nn.BatchNorm1d(256)
        self.relu_fc1 = nn.ReLU()
        self.drop_fc1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.drop1(self.pool1(self.relu1(self.bn1(self.conv1(x)))))
        x = self.drop2(self.pool2(self.relu2(self.bn2(self.conv2(x)))))
        x = self.drop3(self.relu3(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.drop_fc1(self.relu_fc1(self.bn_fc1(self.fc1(x))))
        x = self.fc2(x)
        return x


def generate_html_report(history: Dict[str, List[float]], class_names: List[str], metrics: Dict[str, Any]) -> str:
    """Generates a premium, glassmorphic dark HTML report of model performance."""
    class_names_json = json.dumps(class_names)
    history_json = json.dumps(history)
    report_cleaned = "<br>".join(metrics["class_report"].split("\n"))

    accuracy = metrics["accuracy"] * 100
    precision = metrics["precision_weighted"]
    recall = metrics["recall_weighted"]
    f1 = metrics["f1_weighted"]
    confusion_matrix_list = metrics["conf_matrix"].tolist()

    html_content = f"""<!DOCTYPE html>
<html lang="en" class="dark">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Super-Charged LeNet CNN - Deep Learning Report</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;700&display=swap" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script>
        tailwind.config = {{
            darkMode: 'class',
            theme: {{
                extend: {{
                    fontFamily: {{
                        sans: ['Outfit', 'sans-serif'],
                        mono: ['JetBrains Mono', 'monospace'],
                    }}
                }}
            }}
        }}
    </script>
    <style>
        body {{
            background: linear-gradient(135deg, #090d16 0%, #111827 100%);
            font-family: 'Outfit', sans-serif;
            color: #f1f5f9;
            min-height: 100vh;
        }}
        .glass-panel {{
            background: rgba(17, 24, 39, 0.6);
            backdrop-filter: blur(20px);
            -webkit-backdrop-filter: blur(20px);
            border: 1px solid rgba(255, 255, 255, 0.05);
            box-shadow: 0 12px 40px 0 rgba(0, 0, 0, 0.5);
        }}
    </style>
</head>
<body class="p-6 md:p-12">
    <header class="max-w-7xl mx-auto mb-10 text-center md:text-left flex flex-col md:flex-row justify-between items-center gap-6 pb-8 border-b border-gray-800/80">
        <div>
            <div class="inline-flex items-center gap-2 px-3 py-1.5 rounded-full bg-emerald-500/10 border border-emerald-500/30 text-emerald-400 text-sm font-semibold mb-3">
                <span class="w-2.5 h-2.5 rounded-full bg-emerald-400 animate-pulse"></span>
                Extremely Powerful LeNet Active
            </div>
            <h1 class="text-4xl md:text-5xl font-extrabold tracking-tight text-white">
                Super-Charged LeNet <span class="bg-gradient-to-r from-emerald-400 via-teal-400 to-indigo-500 bg-clip-text text-transparent">CNN Dashboard</span>
            </h1>
            <p class="mt-2 text-slate-400 text-base md:text-lg max-w-2xl font-light">
                Comprehensive training stats, learning curves, and full multi-class evaluations for your Bengali character dataset.
            </p>
        </div>
        <div class="flex gap-4">
            <div class="glass-panel px-6 py-4 rounded-2xl flex flex-col items-center">
                <span class="text-xs font-semibold text-slate-400 uppercase tracking-widest">Final Accuracy</span>
                <span class="text-3xl font-extrabold text-emerald-400 mt-1">{accuracy:.2f}%</span>
            </div>
            <div class="glass-panel px-6 py-4 rounded-2xl flex flex-col items-center">
                <span class="text-xs font-semibold text-slate-400 uppercase tracking-widest">F1-Score</span>
                <span class="text-3xl font-extrabold text-teal-400 mt-1">{f1:.4f}</span>
            </div>
        </div>
    </header>

    <main class="max-w-7xl mx-auto space-y-10">
        <section class="grid grid-cols-1 sm:grid-cols-2 lg:grid-cols-4 gap-6">
            <div class="glass-panel p-6 rounded-2xl">
                <p class="text-xs font-bold text-slate-400 uppercase tracking-wider">Test Loss</p>
                <h3 class="text-2xl font-extrabold text-red-400 mt-2">{history['val_loss'][-1]:.4f}</h3>
            </div>
            <div class="glass-panel p-6 rounded-2xl">
                <p class="text-xs font-bold text-slate-400 uppercase tracking-wider">Weighted Precision</p>
                <h3 class="text-2xl font-extrabold text-sky-400 mt-2">{precision:.4f}</h3>
            </div>
            <div class="glass-panel p-6 rounded-2xl">
                <p class="text-xs font-bold text-slate-400 uppercase tracking-wider">Weighted Recall</p>
                <h3 class="text-2xl font-extrabold text-purple-400 mt-2">{recall:.4f}</h3>
            </div>
            <div class="glass-panel p-6 rounded-2xl">
                <p class="text-xs font-bold text-slate-400 uppercase tracking-wider">Total Parameters</p>
                <h3 class="text-2xl font-extrabold text-amber-400 mt-2">~358,000</h3>
            </div>
        </section>

        <section class="grid grid-cols-1 lg:grid-cols-2 gap-8">
            <div class="glass-panel p-6 rounded-2xl">
                <h3 class="text-lg font-bold text-slate-200 mb-4">📈 Loss Progression</h3>
                <div class="relative w-full h-[350px]">
                    <canvas id="lossChart"></canvas>
                </div>
            </div>
            <div class="glass-panel p-6 rounded-2xl">
                <h3 class="text-lg font-bold text-slate-200 mb-4">🎯 Accuracy Progression</h3>
                <div class="relative w-full h-[350px]">
                    <canvas id="accuracyChart"></canvas>
                </div>
            </div>
        </section>

        <section class="grid grid-cols-1 lg:grid-cols-12 gap-8">
            <div class="glass-panel p-6 rounded-2xl lg:col-span-5 flex flex-col">
                <h3 class="text-lg font-bold text-slate-200 mb-4">📄 Multi-Class Evaluation Report</h3>
                <pre class="bg-gray-950/80 border border-gray-800 rounded-xl p-5 overflow-auto font-mono text-sm text-emerald-400 shadow-inner flex-grow h-[400px]">{report_cleaned}</pre>
            </div>
            <div class="glass-panel p-6 rounded-2xl lg:col-span-7">
                <h3 class="text-lg font-bold text-slate-200 mb-4">🧭 Interactive Confusion Matrix Heatmap</h3>
                <div class="bg-gray-950/40 border border-gray-800 rounded-xl p-6 h-[400px] overflow-auto flex items-center justify-center">
                    <div id="matrixContainer" class="grid gap-1 w-full max-w-md aspect-square"></div>
                </div>
            </div>
        </section>
    </main>

    <footer class="max-w-7xl mx-auto mt-16 text-center text-slate-600 text-sm border-t border-gray-800/40 pt-8">
        ⚡ Super-Charged LeNet CNN Engine • Built with PyTorch & Google Colab.
    </footer>

    <script>
        const historyData = {history_json};
        const classNames = {class_names_json};
        const confMatrix = {json.dumps(confusion_matrix_list)};
        const epochs = Array.from({{length: historyData.train_loss.length}}, (_, i) => i + 1);

        const lossCtx = document.getElementById('lossChart').getContext('2d');
        new Chart(lossCtx, {{
            type: 'line',
            data: {{
                labels: epochs,
                datasets: [
                    {{
                        label: 'Training Loss',
                        data: historyData.train_loss,
                        borderColor: '#f87171',
                        backgroundColor: 'rgba(248, 113, 113, 0.1)',
                        tension: 0.3,
                        fill: true
                    }},
                    {{
                        label: 'Validation Loss',
                        data: historyData.val_loss,
                        borderColor: '#fb7185',
                        backgroundColor: 'rgba(251, 113, 133, 0.05)',
                        tension: 0.3,
                        borderDash: [5, 5]
                    }}
                ]
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: false,
                scales: {{
                    y: {{ grid: {{ color: 'rgba(255,255,255,0.03)' }}, ticks: {{ color: '#94a3b8' }} }},
                    x: {{ grid: {{ display: false }}, ticks: {{ color: '#94a3b8' }} }}
                }},
                plugins: {{
                    legend: {{ labels: {{ color: '#f1f5f9' }} }}
                }}
            }}
        }});

        const accCtx = document.getElementById('accuracyChart').getContext('2d');
        new Chart(accCtx, {{
            type: 'line',
            data: {{
                labels: epochs,
                datasets: [
                    {{
                        label: 'Training Accuracy (%)',
                        data: historyData.train_acc.map(a => a * 100),
                        borderColor: '#34d399',
                        backgroundColor: 'rgba(52, 211, 153, 0.1)',
                        tension: 0.3,
                        fill: true
                    }},
                    {{
                        label: 'Validation Accuracy (%)',
                        data: historyData.val_acc.map(a => a * 100),
                        borderColor: '#60a5fa',
                        backgroundColor: 'rgba(96, 165, 250, 0.05)',
                        tension: 0.3,
                        borderDash: [5, 5]
                    }}
                ]
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: false,
                scales: {{
                    y: {{ grid: {{ color: 'rgba(255,255,255,0.03)' }}, ticks: {{ color: '#94a3b8' }}, min: 0, max: 100 }},
                    x: {{ grid: {{ display: false }}, ticks: {{ color: '#94a3b8' }} }}
                }},
                plugins: {{
                    legend: {{ labels: {{ color: '#f1f5f9' }} }}
                }}
            }}
        }});

        const container = document.getElementById('matrixContainer');
        const size = confMatrix.length;
        container.style.gridTemplateColumns = `repeat(${{size}}, minmax(0, 1fr))`;

        let maxVal = 1;
        confMatrix.forEach(row => row.forEach(val => {{ if(val > maxVal) maxVal = val; }}));

        for (let i = 0; i < size; i++) {{
            for (let j = 0; j < size; j++) {{
                const val = confMatrix[i][j];
                const ratio = val / maxVal;
                const cell = document.createElement('div');
                cell.className = "flex flex-col items-center justify-center p-1 font-mono text-xs rounded border border-gray-950 transition-colors";
                cell.style.backgroundColor = `rgba(52, 211, 153, ${{Math.max(0.04, ratio * 0.85)}})`;
                cell.style.color = ratio > 0.45 ? "#000000" : "#ffffff";
                cell.title = `Actual: ${{classNames[i]}} -> Predicted: ${{classNames[j]}} (${{val}})`;

                const valLabel = document.createElement('span');
                valLabel.className = "font-bold text-sm";
                valLabel.innerText = val;
                cell.appendChild(valLabel);
                container.appendChild(cell);
            }}
        }}
    </script>
</body>
</html>
"""
    return html_content


def main():
    print("=========================================================")
    print("⚡ Super-Charged LeNet CNN Handwritten Character Classifier")
    print("=========================================================")

    try:
        data_path = upload_and_extract_zip()
    except Exception as e:
        print(f"\n❌ Error during file upload/extraction: {e}")
        return

    train_dir = data_path / "train"
    test_dir = data_path / "test"

    if not train_dir.exists() or not test_dir.exists():
        subdirs = [d for d in data_path.iterdir() if d.is_dir()]
        for subdir in subdirs:
            if (subdir / "train").exists() and (subdir / "test").exists():
                train_dir = subdir / "train"
                test_dir = subdir / "test"
                break
        else:
            raise FileNotFoundError("❌ Could not locate 'train' and 'test' folders inside the zip file.")

    class_names = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    print("🔄 Initializing datasets...")
    train_dataset = BengaliDataset(train_dir, class_names, class_to_idx)
    test_dataset = BengaliDataset(test_dir, class_names, class_to_idx)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print(f"📊 Loaded Train dataset: {len(train_dataset)} images | Test dataset: {len(test_dataset)} images")

    model = SuperChargedLeNet(num_classes=len(class_names)).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": []
    }

    print("\n🚀 Commencing training of Super-Charged LeNet model...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_dataset)
        epoch_train_acc = correct_train / total_train

        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = running_val_loss / len(test_dataset)
        epoch_val_acc = correct_val / total_val

        scheduler.step(epoch_val_loss)

        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)

        print(f"Epoch [{epoch+1}/{EPOCHS}] -> Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

        if epoch_val_acc >= 1.0 and epoch_train_acc >= 0.999:
            print("👑 Halted early: Validation accuracy has successfully converged to 100%!")
            break

    print("\n📊 Computing final evaluation metrics...")
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    conf_m = confusion_matrix(all_labels, all_preds)
    class_rep = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)

    metrics = {
        "accuracy": float(acc),
        "precision_weighted": float(precision),
        "recall_weighted": float(recall),
        "f1_weighted": float(f1),
        "conf_matrix": conf_m,
        "class_report": class_rep
    }

    html_report = generate_html_report(history, class_names, metrics)
    report_filename = "lenet_cnn_report.html"

    with open(report_filename, "w", encoding="utf-8") as f:
        f.write(html_report)

    print("\n🏆 Model training and evaluations completed!")
    print(f"📁 Dashboard saved successfully to: '{report_filename}'")

    if IN_COLAB:
        print("📥 Downloading dashboard HTML file to your computer...")
        files.download(report_filename)
        print("✅ Finished.")
    else:
        print("💡 Open 'lenet_cnn_report.html' in your browser to inspect the results.")


if __name__ == "__main__":
    main()


🖥️ Using Device: cpu (CPU Mode - GPU recommended!)
⚡ Super-Charged LeNet CNN Handwritten Character Classifier
💡 [STEP 1] Upload your dataset zip file...


Saving Dataset.zip to Dataset (1).zip
🔄 Extracting dataset to /content/bengali_character_data...
✅ Extraction complete.
🔄 Initializing datasets...
📊 Loaded Train dataset: 12000 images | Test dataset: 3000 images

🚀 Commencing training of Super-Charged LeNet model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/25] -> Train Loss: 2.1966 | Train Acc: 45.85% | Val Loss: 0.9960 | Val Acc: 76.10%
Epoch [2/25] -> Train Loss: 0.9617 | Train Acc: 74.43% | Val Loss: 0.5538 | Val Acc: 85.70%
Epoch [3/25] -> Train Loss: 0.6563 | Train Acc: 81.36% | Val Loss: 0.4131 | Val Acc: 88.80%
Epoch [4/25] -> Train Loss: 0.5117 | Train Acc: 85.37% | Val Loss: 0.3386 | Val Acc: 90.90%
Epoch [5/25] -> Train Loss: 0.4272 | Train Acc: 87.66% | Val Loss: 0.2952 | Val Acc: 91.47%
Epoch [6/25] -> Train Loss: 0.3753 | Train Acc: 88.58% | Val Loss: 0.2539 | Val Acc: 92.90%
Epoch [7/25] -> Train Loss: 0.3268 | Train Acc: 90.04% | Val Loss: 0.2340 | Val Acc: 92.93%
Epoch [8/25] -> Train Loss: 0.2809 | Train Acc: 91.37% | Val Loss: 0.2100 | Val Acc: 93.53%
Epoch [9/25] -> Train Loss: 0.2512 | Train Acc: 92.14% | Val Loss: 0.2092 | Val Acc: 94.00%
Epoch [10/25] -> Train Loss: 0.2385 | Train Acc: 92.58% | Val Loss: 0.2168 | Val Acc: 93.57%
Epoch [11/25] -> Train Loss: 0.2149 | Train Acc: 93.27% | Val Loss: 0.2098 | Va

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Finished.
